# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. `5 (2, 3)`: `arange(6)` раскладывается в матрицу 2×3 `[[0,1,2],[3,4,5]]`, элемент в строке 1 и столбце 2 равен 5.
2. `1 8`: максимум равен 8, а `argmax` возвращает индекс **первого** максимума, то есть 1.
3. `99`: срез `a[::3]` — это view (вид) на тот же массив, а не копия, поэтому изменение `b[0]` меняет и `a[0]`.
4. `2 0.5`: маска `[F, T, T, F]`; `True` считается как 1, сумма равна 2, среднее 2/4 = 0.5.
5. `(2, 3) 0.25`: `zeros((2, 3))` имеет форму (2, 3), а `linspace(0, 1, 5)` даёт `[0, 0.25, 0.5, 0.75, 1]`.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
# База
# 1. Самый дорогой заказ
i = amount.argmax()
print("Позиция:", i)
print("Сумма:", amount[i], "| клиент:", cust[i], "| город:", city[i])
print(f"Самый дорогой заказ — {amount[i]:,.0f} ₸, клиент {cust[i]} из города {city[i]}. "
      "Это тестовый клиент, а не настоящий покупатель: его заказы нужно убирать из расчётов.")

# 2. Выручка оплаченных и сумма неоплаченных
paid_sum = amount[paid].sum()
unpaid_sum = amount[~paid].sum()
print(f"Оплачено: {paid_sum:,.0f} ₸, не оплачено: {unpaid_sum:,.0f} ₸")
print(f"Магазин получил {paid_sum:,.0f} ₸, ещё {unpaid_sum:,.0f} ₸ пришлось на отменённые заказы.")

# 3. Заказы в Шымкенте
shym = city == "Шымкент"
print("Заказов в Шымкенте:", shym.sum())
print(f"Доля: {shym.mean():.1%}")
print(f"В Шымкенте {shym.sum()} заказов — это {shym.mean():.1%} от всех заказов.")

Позиция: 1001
Сумма: 9999990.0 | клиент: TEST | город: Алматы
Самый дорогой заказ — 9,999,990 ₸, клиент TEST из города Алматы. Это тестовый клиент, а не настоящий покупатель: его заказы нужно убирать из расчётов.
Оплачено: 101,909,360 ₸, не оплачено: 7,974,000 ₸
Магазин получил 101,909,360 ₸, ещё 7,974,000 ₸ пришлось на отменённые заказы.
Заказов в Шымкенте: 804
Доля: 14.7%
В Шымкенте 804 заказов — это 14.7% от всех заказов.


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [ ]:
# Стандарт
# 4. Скидка 15% на оплаченные заказы дороже 50 000 ₸
big = paid & (amount > 50000)
loss_all = (amount[big] * 0.15).sum()

big_no_test = big & (cust != "TEST")
loss_no_test = (amount[big_no_test] * 0.15).sum()

print(f"Со всеми заказами: {loss_all:,.0f} ₸")
print(f"Без клиента TEST:  {loss_no_test:,.0f} ₸")
print("Заказов дороже 50 000 у TEST:", (big & (cust == "TEST")).sum(), "из", big.sum())
print(f"Реально магазин недополучит около {loss_no_test:,.0f} ₸, а не {loss_all:,.0f} ₸.")

# 5. Доля заказов, доставленных за 1–2 дня, по каждому городу
for c in np.unique(city):
    d = days[(city == c) & ~np.isnan(days)]   # заказы без срока доставки не считаем
    fast = (d >= 1) & (d <= 2)
    print(f"{c:<10} {fast.mean():.1%}")
print("В Шымкенте за 1–2 дня доставляют только ~31% заказов, а в остальных городах 63–71%: проблема с доставкой именно там.")

# 6. Исправленный средний чек: только оплаченные и без TEST
good = paid & (cust != "TEST")
avg_wrong = amount.mean()
avg = amount[good].mean()
print("Неправильно:", round(avg_wrong))
print("Средний чек:", round(avg))
print(f"Средний чек оплаченного заказа — {avg:,.0f} ₸.")

Со всеми заказами: 5,702,422 ₸
Без клиента TEST:  1,202,427 ₸
Заказов дороже 50 000 у TEST: 3 из 127
Реально магазин недополучит около 1,202,427 ₸, а не 5,702,422 ₸.
Актобе     62.8%
Алматы     66.5%
Астана     65.4%
Атырау     66.7%
Караганда  71.0%
Шымкент    30.7%
В Шымкенте за 1–2 дня доставляют только ~31% заказов, а в остальных городах 63–71%: проблема с доставкой именно там.
Неправильно: 20114
Средний чек: 14672
Средний чек оплаченного заказа — 14,672 ₸.


**Ответ к задаче 4.** Без TEST скидка обходится примерно в 1,2 млн ₸, а с ним — в 5,7 млн ₸, почти в 5 раз больше. У TEST всего 3 заказа дороже 50 000 ₸, но они огромные (до 9 999 990 ₸), и 15% от них перекрывают все настоящие заказы. Это тестовые данные, их нужно убирать.

**Ответ к задаче 6.** Две ошибки: (1) `amount.mean()` учитывает отменённые заказы, хотя средний чек считают только по оплаченным; (2) в данные попал клиент TEST с гигантскими суммами, и он сильно завышает среднее. Правильно так: `amount[paid & (cust != "TEST")].mean()`, получается около 14 672 ₸ вместо 20 114 ₸.

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж